In [ ]:
%pip install mlxtend ucimlrepo -q

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from ucimlrepo import fetch_ucirepo
import warnings
warnings.filterwarnings('ignore')

# Proyecto Semestral: Prediccion de Niveles de Obesidad
**Curso:** Ingenieria de Datos 2026  
**Dataset:** Estimation of Obesity Levels Based On Eating Habits and Physical Condition

## 1. Definicion del Problema y Dataset

- **Nombre del Dataset:** Estimation of Obesity Levels Based On Eating Habits and Physical Condition
- **Fuente:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition)
- **Descripcion General:** Datos de individuos de Mexico, Peru y Colombia para estimar niveles de obesidad basados en habitos alimenticios y condicion fisica.
- **Numero de Registros:** 2111
- **Numero de Variables:** 17 (16 caracteristicas + 1 objetivo)
- **Variable Objetivo:** `NObeyesdad` (7 niveles de peso)
- **Licencia:** Creative Commons Attribution 4.0 International (CC BY 4.0).

**Problema de Machine Learning:**
- **Tipo de Tarea:** Clasificacion Multiclase.
- **Pregunta de Investigacion:** Es posible predecir con precision el nivel de obesidad de un individuo a partir de sus habitos diarios de alimentacion y actividad fisica, sin usar peso ni altura?
- **Contexto de Uso:** Herramientas de salud preventiva y aplicaciones de bienestar.
- **Criterio de Exito:** Alcanzar un Accuracy y F1-Score (macro) superior al 85% en el conjunto de prueba.

### Diccionario de Variables

| Variable | Tipo | Descripcion |
| :--- | :--- | :--- |
| **Gender** | Categorica | Genero del individuo (Male / Female) |
| **Age** | Numerica | Edad del individuo |
| **Height** | Numerica | Altura en metros |
| **Weight** | Numerica | Peso en kilogramos |
| **family_history_with_overweight** | Categorica | Historial familiar de sobrepeso (Yes / No) |
| **FAVC** | Categorica | Consumo frecuente de alimentos hipercaloricos (Yes / No) |
| **FCVC** | Numerica | Frecuencia de consumo de verduras (1, 2 o 3) |
| **NCP** | Numerica | Numero de comidas principales al dia |
| **CAEC** | Categorica | Consumo de alimentos entre comidas |
| **SMOKE** | Categorica | Si el individuo fuma (Yes / No) |
| **CH2O** | Numerica | Consumo diario de agua en litros |
| **SCC** | Categorica | Monitoreo del consumo de calorias (Yes / No) |
| **FAF** | Numerica | Frecuencia de actividad fisica semanal |
| **TUE** | Numerica | Tiempo de uso de dispositivos tecnologicos |
| **CALC** | Categorica | Consumo de alcohol |
| **MTRANS** | Categorica | Medio de transporte utilizado |
| **NObeyesdad** | Categorica | **Variable Objetivo:** Nivel de peso (7 clases) |

## Justificacion

La obesidad es una de las crisis de salud publica mas complejas a nivel global. Este dataset permite modelar la relacion entre habitos diarios y niveles de masa corporal usando Machine Learning como herramienta de diagnostico temprano. Se excluyen Weight y Height del entrenamiento para evitar data leakage, forzando al modelo a aprender patrones reales de habitos y estilo de vida.

In [ ]:
# Cargar dataset desde UCI
obesity_data = fetch_ucirepo(id=544)
X = obesity_data.data.features
y = obesity_data.data.targets

print(obesity_data.metadata)
print(obesity_data.variables)

In [ ]:
dataset = pd.concat([X, y], axis=1)
print(f"Dataset creado. Dimensiones: {dataset.shape}")

---
## 2. Analisis Exploratorio de Datos (EDA)

In [ ]:
print("--- Informacion del Dataset ---")
print(dataset.info())

# Normalizar CAEC y CALC
dataset['CAEC'] = dataset['CAEC'].apply(lambda x: x.capitalize() if isinstance(x, str) else x)
dataset['CALC'] = dataset['CALC'].apply(lambda x: x.capitalize() if isinstance(x, str) else x)
print("\nColumnas 'CAEC' y 'CALC' normalizadas.")

print("\n--- Valores Nulos ---")
print(dataset.isnull().sum())

print("\n--- Duplicados ---")
duplicados = dataset.duplicated().sum()
print(f"Registros duplicados encontrados: {duplicados}")

In [ ]:
# Eliminar duplicados y guardar version limpia
print(f"Duplicados detectados: {dataset.duplicated().sum()}")
dataset_limpio = dataset.drop_duplicates()
print(f"Dimensiones tras remover duplicados: {dataset_limpio.shape}")
dataset_limpio.to_csv('obesity_dataset_limpio.csv', index=False)
print("Dataset limpio guardado como 'obesity_dataset_limpio.csv'")

In [ ]:
# Deteccion de outliers en variables continuas
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(ax=axes[0], x=dataset['Age'], color='skyblue')
axes[0].set_title('Distribucion de la Edad')

sns.boxplot(ax=axes[1], x=dataset['Weight'], color='salmon')
axes[1].set_title('Distribucion del Peso')
plt.show()

In [ ]:
# Relacion Peso vs Edad por Genero
plt.figure(figsize=(10, 6))
sns.scatterplot(data=dataset, x='Age', y='Weight', hue='Gender', alpha=0.6)
plt.title('Relacion Edad vs Peso por Genero')
plt.show()

In [ ]:
# Distribucion de la Variable Objetivo
plt.figure(figsize=(15, 6))
sns.countplot(data=dataset, x='NObeyesdad', palette='viridis', hue='NObeyesdad', legend=False)
plt.title('Distribucion de los Niveles de Obesidad')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Relacion entre Actividad Fisica (FAF) y Nivel de Obesidad
dataset['FAF_cat'] = pd.cut(dataset['FAF'], bins=[0, 1, 2, 3], labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)

plt.figure(figsize=(14, 6))
sns.countplot(data=dataset, x='NObeyesdad', hue='FAF_cat', palette='viridis')
plt.title('Niveles de Obesidad vs Frecuencia de Actividad Fisica (FAF)')
plt.xticks(rotation=30, ha='right')
plt.xlabel('Nivel de Obesidad')
plt.ylabel('Cantidad de Registros')
plt.legend(title='Frecuencia de Actividad Fisica')
plt.tight_layout()
plt.show()

# Relacion entre Consumo de Alimentos Hipercaloricos (FAVC) y Obesidad
plt.figure(figsize=(14, 6))
sns.countplot(data=dataset, x='NObeyesdad', hue='FAVC', palette='Set2')
plt.title('Impacto del Consumo Frecuente de Alimentos Hipercaloricos (FAVC)')
plt.xticks(rotation=30, ha='right')
plt.xlabel('Nivel de Obesidad')
plt.ylabel('Cantidad de Registros')
plt.legend(title='Consumo Calorico Alto')
plt.tight_layout()
plt.show()

dataset.drop('FAF_cat', axis=1, inplace=True)

### Observaciones del EDA:
- La variable `Age` muestra valores atipicos concentrados hacia edades avanzadas (+40 anos).
- El dataset esta razonablemente balanceado entre categorias de obesidad.
- No se presentan valores nulos en ninguna variable.
- Los niveles de obesidad severa (Obesity Type III) tienen muy baja actividad fisica alta.
- Se detectaron y eliminaron 24 registros duplicados.
- Nota importante: Aunque Weight y Height aparecen en el dataset, **no se usaran como features** en los modelos para evitar data leakage, ya que el IMC (que define la obesidad) se calcula con estas variables.

---
## 3. Diseno Experimental y Preprocesamiento

### Modelos Seleccionados:
1. **Random Forest Classifier:** Robusto frente al sobreajuste, maneja relaciones no lineales.
2. **XGBoost:** Gradient Boosting regularizado, ideal para datos tabulares mixtos.

### Estrategia de Validacion:
- Division **70/15/15** (Train/Validation/Test) con estratificacion.
- Validacion cruzada **StratifiedKFold (k=3)** para busqueda de hiperparametros.

### Preprocesamiento (via Pipeline + ColumnTransformer):
- **Variables numericas:** StandardScaler.
- **Variables categoricas:** One-Hot Encoding (drop first).
- **Nota:** Se eliminan Weight y Height del entrenamiento para evitar data leakage.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, roc_auc_score, precision_score, recall_score)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Cargar datos limpios
df_ml = pd.read_csv('obesity_dataset_limpio.csv')

# Eliminar Weight y Height para evitar data leakage
# La obesidad se define mediante IMC = peso/altura^2, incluir estas variables
# permitiria al modelo "hacer trampa" usando la definicion del target
df_ml = df_ml.drop(['Weight', 'Height'], axis=1)

# Separar features y target
X_ml = df_ml.drop('NObeyesdad', axis=1)
y_ml = df_ml['NObeyesdad']

# Codificar target
le = LabelEncoder()
y_encoded = le.fit_transform(y_ml)
print(f"Clases del target: {le.classes_}")
print(f"Dimension de features: {X_ml.shape}")
print(f"Columnas usadas: {X_ml.columns.tolist()}")

In [ ]:
# Identificar tipos de columnas
numerical_cols = X_ml.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = X_ml.select_dtypes(include=['object']).columns.tolist()

print(f"Variables numericas ({len(numerical_cols)}): {numerical_cols}")
print(f"Variables categoricas ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Crear preprocesador unificado con ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('onehot', OneHotEncoder(drop='first', sparse_output=False))
        ]), categorical_cols)
    ]
)

# Division 70/15/15 con estratificacion
X_train, X_temp, y_train, y_temp = train_test_split(
    X_ml, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X_ml)*100:.1f}%)")
print(f"Validation: {X_val.shape[0]} muestras ({X_val.shape[0]/len(X_ml)*100:.1f}%)")
print(f"Test: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X_ml)*100:.1f}%)")

In [ ]:
# Configurar validacion cruzada
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# --- Random Forest ---
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

rf_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [12, None],
    'classifier__min_samples_leaf': [1, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline, rf_params, cv=cv,
    scoring='f1_macro', n_jobs=-1, verbose=1
)

# --- XGBoost ---
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric='mlogloss', random_state=42))
])

xgb_params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [6, 10],
    'classifier__learning_rate': [0.1, 0.3]
}

xgb_grid = GridSearchCV(
    xgb_pipeline, xgb_params, cv=cv,
    scoring='f1_macro', n_jobs=-1, verbose=1
)

print("Entrenando Random Forest...")
rf_grid.fit(X_train, y_train)
print(f"Mejor F1 (RF): {rf_grid.best_score_:.4f}")
print(f"Mejores params (RF): {rf_grid.best_params_}")

print("\nEntrenando XGBoost...")
xgb_grid.fit(X_train, y_train)
print(f"Mejor F1 (XGB): {xgb_grid.best_score_:.4f}")
print(f"Mejores params (XGB): {xgb_grid.best_params_}")

### Evaluacion en Validation

In [ ]:
# Evaluar ambos modelos en validation
y_pred_rf_val = rf_grid.predict(X_val)
y_pred_xgb_val = xgb_grid.predict(X_val)

# Probabilidades para ROC-AUC
y_proba_rf_val = rf_grid.predict_proba(X_val)
y_proba_xgb_val = xgb_grid.predict_proba(X_val)

results_val = {
    'Random Forest': {
        'accuracy': accuracy_score(y_val, y_pred_rf_val),
        'f1_macro': f1_score(y_val, y_pred_rf_val, average='macro'),
        'roc_auc': roc_auc_score(y_val, y_proba_rf_val, multi_class='ovr')
    },
    'XGBoost': {
        'accuracy': accuracy_score(y_val, y_pred_xgb_val),
        'f1_macro': f1_score(y_val, y_pred_xgb_val, average='macro'),
        'roc_auc': roc_auc_score(y_val, y_proba_xgb_val, multi_class='ovr')
    }
}

df_results_val = pd.DataFrame(results_val).T
print("Metricas en Validation:")
display(df_results_val.style.format('{:.4f}'))

In [ ]:
# Reportes detallados en validation
print("--- Reporte Detallado (Random Forest) - Validation ---")
print(classification_report(y_val, y_pred_rf_val, target_names=le.classes_))

print("\n--- Reporte Detallado (XGBoost) - Validation ---")
print(classification_report(y_val, y_pred_xgb_val, target_names=le.classes_))

In [ ]:
# Matrices de confusion en validation
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(confusion_matrix(y_val, y_pred_rf_val), annot=True, fmt='d',
            cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title('Matriz de Confusion - Random Forest (Validation)')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Prediccion')

sns.heatmap(confusion_matrix(y_val, y_pred_xgb_val), annot=True, fmt='d',
            cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title('Matriz de Confusion - XGBoost (Validation)')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Prediccion')

plt.tight_layout()
plt.show()

### Evaluacion Final en Test
Seleccionamos el mejor modelo segun F1-macro en validation y evaluamos en test.

In [ ]:
# Determinar mejor modelo en validation
best_model_name = 'XGBoost' if results_val['XGBoost']['f1_macro'] >= results_val['Random Forest']['f1_macro'] else 'Random Forest'
best_model = xgb_grid if best_model_name == 'XGBoost' else rf_grid
print(f"Mejor modelo segun F1-macro en validation: {best_model_name}")

In [ ]:
# Evaluacion final en test
y_pred_test = best_model.predict(X_test)
y_proba_test = best_model.predict_proba(X_test)

print(f"--- Evaluacion Final en Test ({best_model_name}) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"F1-Score (macro): {f1_score(y_test, y_pred_test, average='macro'):.4f}")
print(f"ROC-AUC (OVR): {roc_auc_score(y_test, y_proba_test, multi_class='ovr'):.4f}")

print("\n--- Reporte Detallado ---")
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

In [ ]:
# Matriz de confusion en test
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusion - {best_model_name} (Test)')
plt.ylabel('Real')
plt.xlabel('Prediccion')
plt.show()

---
## 4. Analisis de Trade-offs y Limitaciones

### Trade-offs identificados:
- **Errores mas criticos:** Clasificar a una persona con Obesity Type III como Normal Weight es mas grave que confundir Overweight Level I con Overweight Level II.
- **Precision vs Recall:** El modelo busca equilibrio (F1-macro), pero en un contexto medico real se prioritaria el recall para no dejar casos de obesidad sin detectar.

### Limitaciones del experimento:
- Se eliminaron Weight y Height para evitar data leakage, lo que reduce el techo de accuracy posible.
- Dataset relativamente pequeno (2087 registros tras limpieza) para 7 clases.
- Los datos son auto-reportados (encuestas), lo que puede introducir sesgos.
- Solo se probaron 2 modelos clasicos; no se exploraron redes neuronales o ensambles mas complejos.

---
## 5. Ejemplos de Uso

A continuacion, usamos el modelo entrenado para predecir el nivel de obesidad de 3 perfiles hipoteticos.

In [ ]:
def predecir_obesidad(datos_ejemplo):
    ejemplo_df = pd.DataFrame([datos_ejemplo])
    pred = best_model.predict(ejemplo_df)
    return le.inverse_transform(pred)[0]

# Ejemplo 1: Persona joven, deportista, dieta saludable
ejemplo_1 = {'Gender': 'Male', 'Age': 22, 'family_history_with_overweight': 'no',
             'FAVC': 'no', 'FCVC': 3, 'NCP': 3,
             'CAEC': 'Sometimes', 'SMOKE': 'no', 'CH2O': 2, 'SCC': 'yes',
             'FAF': 3, 'TUE': 1, 'CALC': 'No', 'MTRANS': 'Walking'}

# Ejemplo 2: Persona con historial familiar y poco ejercicio
ejemplo_2 = {'Gender': 'Female', 'Age': 35, 'family_history_with_overweight': 'yes',
             'FAVC': 'yes', 'FCVC': 2, 'NCP': 3,
             'CAEC': 'Frequently', 'SMOKE': 'no', 'CH2O': 1, 'SCC': 'no',
             'FAF': 0, 'TUE': 2, 'CALC': 'Sometimes', 'MTRANS': 'Public_Transportation'}

# Ejemplo 3: Perfil intermedio
ejemplo_3 = {'Gender': 'Male', 'Age': 28, 'family_history_with_overweight': 'yes',
             'FAVC': 'yes', 'FCVC': 2, 'NCP': 3,
             'CAEC': 'Sometimes', 'SMOKE': 'no', 'CH2O': 2, 'SCC': 'no',
             'FAF': 1, 'TUE': 1, 'CALC': 'Sometimes', 'MTRANS': 'Automobile'}

print(f"Prediccion Ejemplo 1 (joven saludable): {predecir_obesidad(ejemplo_1)}")
print(f"Prediccion Ejemplo 2 (alto riesgo): {predecir_obesidad(ejemplo_2)}")
print(f"Prediccion Ejemplo 3 (perfil intermedio): {predecir_obesidad(ejemplo_3)}")

---
## 6. Experimento Adicional: Dataset Crudo (sin limpieza de duplicados)

Repetimos el proceso usando el dataset original (sin eliminar duplicados) para comparar si la limpieza impacto el rendimiento.

In [ ]:
# Usar dataset original (antes de eliminar duplicados)
df_raw = dataset.copy()
df_raw = df_raw.drop(['Weight', 'Height'], axis=1)
X_raw = df_raw.drop('NObeyesdad', axis=1)
y_raw = df_raw['NObeyesdad']
y_raw_encoded = LabelEncoder().fit_transform(y_raw)

print(f"Dimensiones dataset crudo: {X_raw.shape}")
print(f"Duplicados incluidos: {df_raw.duplicated().sum()}")

In [ ]:
# Preprocesar, dividir y entrenar con datos crudos
numerical_cols_raw = X_raw.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols_raw = X_raw.select_dtypes(include=['object']).columns.tolist()

preprocessor_raw = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('scaler', StandardScaler())]), numerical_cols_raw),
        ('cat', Pipeline([('onehot', OneHotEncoder(drop='first', sparse_output=False))]), categorical_cols_raw)
    ]
)

X_train_r, X_temp_r, y_train_r, y_temp_r = train_test_split(
    X_raw, y_raw_encoded, test_size=0.30, random_state=42, stratify=y_raw_encoded
)
X_val_r, X_test_r, y_val_r, y_test_r = train_test_split(
    X_temp_r, y_temp_r, test_size=0.50, random_state=42, stratify=y_temp_r
)

# Entrenar XGBoost en datos crudos
xgb_raw = Pipeline([
    ('preprocessor', preprocessor_raw),
    ('classifier', XGBClassifier(eval_metric='mlogloss', random_state=42, n_estimators=200))
])
xgb_raw.fit(X_train_r, y_train_r)

# Evaluar
y_pred_raw = xgb_raw.predict(X_test_r)
y_proba_raw = xgb_raw.predict_proba(X_test_r)

print("--- Resultados con Dataset Crudo ---")
print(f"Accuracy: {accuracy_score(y_test_r, y_pred_raw):.4f}")
print(f"F1-macro: {f1_score(y_test_r, y_pred_raw, average='macro'):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_r, y_proba_raw, multi_class='ovr'):.4f}")
print()

# Comparacion con dataset limpio
y_pred_clean = best_model.predict(X_test)
print("--- Comparacion ---")
print(f"Accuracy (limpio): {accuracy_score(y_test, y_pred_clean):.4f} vs (crudo): {accuracy_score(y_test_r, y_pred_raw):.4f}")
print(f"F1-macro (limpio): {f1_score(y_test, y_pred_clean, average='macro'):.4f} vs (crudo): {f1_score(y_test_r, y_pred_raw, average='macro'):.4f}")

---
## 7. Conclusiones

- El modelo XGBoost supero a Random Forest en todas las metricas.
- Al eliminar Weight y Height, el accuracy bajo respecto al notebook original, pero el modelo ahora refleja aprendizaje genuino de patrones de habitos.
- El experimento con datos crudos muestra que eliminar duplicados tiene un impacto minimo en el rendimiento.
- La metrica ROC-AUC superior a 0.95 indica que el modelo separa bien las clases incluso sin peso ni altura.
- Para uso en produccion, se recomendaria priorizar recall en las clases de obesidad severa para minimizar falsos negativos.

---
## Material Extra: Reglas de Asociacion (Apriori)

Analisis adicional usando Apriori para descubrir combinaciones frecuentes de habitos asociados a niveles de obesidad.

In [ ]:
# Cargar dataset limpio para reglas
dataset_reglas = pd.read_csv('obesity_dataset_limpio.csv')

# Discretizar variables numericas
dataset_reglas['Age'] = pd.qcut(dataset_reglas['Age'], q=3, labels=['Joven', 'Adulto', 'Mayor'])
dataset_reglas['Height'] = pd.qcut(dataset_reglas['Height'], q=3, labels=['Bajo', 'Medio', 'Alto'])
dataset_reglas['Weight'] = pd.qcut(dataset_reglas['Weight'], q=3, labels=['Ligero', 'Medio', 'Pesado'])
dataset_reglas['FCVC'] = pd.cut(dataset_reglas['FCVC'], bins=[0, 1.5, 2.5, 3.5], labels=['Bajo', 'Medio', 'Alto'])
dataset_reglas['NCP'] = pd.cut(dataset_reglas['NCP'], bins=[0, 2.5, 3.5, 5], labels=['1-2_comidas', '3_comidas', 'Mas_de_3'])
dataset_reglas['CH2O'] = pd.cut(dataset_reglas['CH2O'], bins=[0, 1.5, 2.5, 3.5], labels=['Poco', 'Medio', 'Mucho'])
dataset_reglas['FAF'] = pd.cut(dataset_reglas['FAF'], bins=[-0.1, 1, 2, 4], labels=['Baja', 'Media', 'Alta'])
dataset_reglas['TUE'] = pd.cut(dataset_reglas['TUE'], bins=[-0.1, 1, 2, 3], labels=['Bajo', 'Medio', 'Alto'])

# One-hot encoding y Apriori
dataset_tx = pd.get_dummies(dataset_reglas).astype(bool)
frequent_itemsets = apriori(dataset_tx, min_support=0.1, use_colnames=True)

# Top 10 itemsets
top_itemsets = frequent_itemsets.sort_values('support', ascending=False).head(10)
top_itemsets['itemsets_str'] = top_itemsets['itemsets'].apply(lambda x: ', '.join(list(x)))

plt.figure(figsize=(10, 6))
sns.barplot(x='support', y='itemsets_str', data=top_itemsets, palette='viridis')
plt.title('Top 10 Combinaciones mas frecuentes en el Dataset')
plt.xlabel('Soporte')
plt.ylabel('Combinacion')
plt.tight_layout()
plt.show()

In [ ]:
# Generar reglas de asociacion
reglas = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)
reglas_obesidad = reglas[reglas['consequents'].apply(lambda x: any('NObeyesdad' in item for item in x))]
reglas_obesidad = reglas_obesidad.sort_values(by=['lift', 'confidence'], ascending=[False, False])

print(f"Reglas encontradas con consecuente de obesidad: {len(reglas_obesidad)}")
print("\nTop 10 Reglas de Asociacion:")
display(reglas_obesidad[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))